# Nuclear targeting: ML playground

Every dataset of the project, one interface. Build the tables first (`make check`, `python -m nucprob.us1985.dataset`, the `nucprob.transfer` steps), then run this notebook with `uv run jupyter lab`.

The sealed test blocks are left out of every dataset unless you pass `include_sealed=True`; they are kept for the project's single final scoring.

In [ ]:
import pandas as pd

from nucprob.playground import ALGORITHMS, evaluate, load, transfer
from nucprob.playground.datasets import names
from nucprob.playground.evaluate import explain

pd.set_option("display.width", 200)
print(names())
print(list(ALGORITHMS))

## 1. A dataset
Soviet bloc towns of 10,000+ around 1956; the target is whether the SAC 1956 study names them.

In [ ]:
ds = load("sac1956")
print(ds.summary())
ds.frame.head()

## 2. The algorithm zoo under spatial cross-validation
300 km blocks, so that neighbouring towns stay on the same side of a split. `scheme='random'` shows how optimistic random folds are; `scheme='country'` leaves one country out at a time.

In [ ]:
board = evaluate(
    ds, ["dummy", "logit", "spline_logit", "knn", "rf", "hgb", "lgbm", "xgb", "catboost"]
)
board[["algorithm", "pr_auc_mean", "pr_auc_std", "roc_auc_mean", "base_rate"]].round(3)

## 3. Which feature families matter?
Compare population alone, population plus each family, and everything but one family.

In [ ]:
rows = []
for fam in ds.families:
    if fam == "population":
        continue
    for label, sub in [
        (f"population + {fam}", ds.select(f"population,{fam}")),
        (f"all but {fam}", ds.select(drop=fam)),
    ]:
        b = evaluate(sub, ["lgbm"], verbose=False)
        rows.append({"features": label, "pr_auc": b["pr_auc_mean"].iloc[0]})
base = evaluate(ds.select("population"), ["lgbm"], verbose=False)["pr_auc_mean"].iloc[0]
print(f"population only: {base:.3f}")
pd.DataFrame(rows).sort_values("pr_auc", ascending=False)

## 4. Explanation
Permutation importance (and mean |SHAP| for boosted trees).

In [ ]:
explain(ds, "lgbm").head(15)

## 5. Transfer: train on one plan, score another
The bloc model scored on the UK's Square Leg exercise, with the features both share.

In [ ]:
train = load("sac1956")
for test_name in [n for n in names() if n.startswith("transfer:")][:6]:
    test = load(test_name)
    b = transfer(train, test, ["logit", "lgbm"], verbose=False)
    print(
        test_name,
        int(test.y.sum()),
        "listed of",
        len(test.frame),
        b[["algorithm", "roc_auc", "pr_auc"]].round(3).to_dict("records"),
    )

## 6. Another country: US counties in NAPB-90 (FEMA, 1987)

In [ ]:
us = load("napb90:very_high")
print(us.summary())
evaluate(us, ["logit", "rf", "lgbm", "catboost"])[
    ["algorithm", "pr_auc_mean", "roc_auc_mean"]
].round(3)

## 7. A map of out-of-fold predictions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from nucprob.playground.algos import algorithms
from nucprob.playground.evaluate import fold_ids

ids = fold_ids(ds)
p = np.full(len(ds.frame), np.nan)
for f in np.unique(ids):
    t = ids == f
    m = algorithms("lgbm")["lgbm"].factory().fit(ds.X[~t], ds.y[~t])
    p[t] = m.predict_proba(ds.X[t])[:, 1]
fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter(ds.frame.lon, ds.frame.lat, c=p, s=6, cmap="magma")
ax.scatter(
    ds.frame.lon[ds.y == 1],
    ds.frame.lat[ds.y == 1],
    s=12,
    facecolors="none",
    edgecolors="cyan",
    linewidths=0.5,
)
plt.colorbar(sc, label="out-of-fold probability")
ax.set_title("SAC 1956: predicted vs listed (cyan)")